In [1]:
!pip install -U transformers datasets huggingface_hub

In [2]:
from huggingface_hub import login

login()

In [3]:
from transformers import AutoTokenizer

model_name = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

template_tokenizer = AutoTokenizer.from_pretrained(
    model_name
)

print("Tokenizer chargé avec succès !")

Tokenizer chargé avec succès !


In [4]:
from datasets import load_dataset

dataset = load_dataset(
    "HuggingFaceH4/ultrachat_200k",
    split="test_sft"
)

dataset = (
    dataset
    .shuffle(seed=42)
    .select(range(3_000))
)

print(dataset)

Dataset({
    features: ['prompt', 'prompt_id', 'messages'],
    num_rows: 3000
})


In [32]:
from trl import SFTTrainer

# ============================================================
# SFTTrainer - API récente de TRL
# ============================================================

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,

    # Configuration d'entraînement
    args=training_arguments,

    # Dans les versions récentes :
    # tokenizer -> processing_class
    processing_class=tokenizer,

    # Configuration LoRA / QLoRA
    peft_config=peft_config,
)

# ============================================================
# Entraînement
# ============================================================

trainer.train()

# ============================================================
# Sauvegarder uniquement les poids LoRA
# ============================================================

trainer.model.save_pretrained(
    "TinyLlama-1.1B-qlora"
)

# Sauvegarder également le tokenizer
tokenizer.save_pretrained(
    "TinyLlama-1.1B-qlora"
)

print("Entraînement terminé.")
print("Modèle QLoRA sauvegardé dans : TinyLlama-1.1B-qlora")

ValueError: You passed a `PeftModel` instance together with a `peft_config` to the trainer. Please first merge and unload the existing adapter, save the resulting base model, and then pass that base model along with the new `peft_config` to the trainer.

In [6]:
# Example of formatted prompt
print(dataset["text"][2576])

<|user|>
Given the text: Knock, knock. Who’s there? Hike.
Can you continue the joke based on the given text material "Knock, knock. Who’s there? Hike"?</s>
<|assistant|>
Sure! Knock, knock. Who's there? Hike. Hike who? Hike up your pants, it's cold outside!</s>
<|user|>
Can you tell me another knock-knock joke based on the same text material "Knock, knock. Who's there? Hike"?</s>
<|assistant|>
Of course! Knock, knock. Who's there? Hike. Hike who? Hike your way over here and let's go for a walk!</s>



In [7]:
!pip install -U bitsandbytes>=0.46.1

In [8]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
 load_in_4bit=True, # Use 4-bit precision model loading
 bnb_4bit_quant_type="nf4", # Quantization type
 bnb_4bit_compute_dtype="float16", # Compute dtype
 bnb_4bit_use_double_quant=True, # Apply nested quantization
)
# Load the model to train on the GPU
model = AutoModelForCausalLM.from_pretrained(
 model_name,
 device_map="auto",
 # Leave this out for regular SFT
 quantization_config=bnb_config,
)
model.config.use_cache = False
model.config.pretraining_tp = 1
# Load LLaMA tokenizer
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

model.safetensors: reconstructing file:   0%|          |  0.00B / 4.40GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/129 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/776 [00:00<?, ?B/s]

tokenizer.model: reconstructing file:   0%|          |  0.00B /  500kB            

tokenizer.model: downloading bytes:           |  0.00B            

tokenizer.json:   0%|          | 0.00/1.84M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/414 [00:00<?, ?B/s]

In [9]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
# Prepare LoRA Configuration
peft_config = LoraConfig(
 lora_alpha=32, # LoRA Scaling
 lora_dropout=0.1, # Dropout for LoRA Layers
 r=64, # Rank
 bias="none",
 task_type="CAUSAL_LM",
 target_modules= # Layers to target
 ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj",
"down_proj"]
)
# Prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [10]:
from transformers import TrainingArguments
output_dir = "./results"
# Training arguments
training_arguments = TrainingArguments(
 output_dir=output_dir,
 per_device_train_batch_size=2,
 gradient_accumulation_steps=4,
 optim="paged_adamw_32bit",
 learning_rate=2e-4,
 lr_scheduler_type="cosine",
 num_train_epochs=1,
 logging_steps=10,
 fp16=True,
 gradient_checkpointing=True
)

In [18]:
!pip install -U trl transformers peft accelerate bitsandbytes datasets

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 30.7 MB/s eta 0:00:00
  Attempting uninstall: peft
    Found existing installation: peft 0.19.1
    Uninstalling peft-0.19.1:
      Successfully uninstalled peft-0.19.1


In [31]:
from trl import SFTTrainer

# ============================================================
# SFTTrainer - API récente de TRL
# ============================================================

# Correction: s'assurer que le pad_token est valide pour le SFTTrainer
tokenizer.pad_token = tokenizer.eos_token

def formatting_func(example):
    # Retourner la chaîne de texte formatée directement
    return example["text"]

# SFTConfig n'est pas utilisé comme argument 'config' direct dans SFTTrainer
# Les arguments qu'il contient (comme max_seq_length, completion_only_loss) sont passés directement.

trainer = SFTTrainer(
    model=model,
    train_dataset=dataset,
    args=training_arguments,
    processing_class=tokenizer,
    formatting_func=formatting_func,
    # dataset_text_field="text", # Spécifie explicitement la colonne de texte pour l'entraînement (argument non supporté ici)
    # max_seq_length=1024, # Cet argument n'est plus pris en charge directement par SFTTrainer
    # completion_only_loss=False, # Désactiver la perte uniquement sur la complétion (argument non supporté ici)
                                # car nous entraînons sur le texte entier formaté dans la colonne 'text'.
    # peft_config=peft_config, # Commenté car le modèle est déjà un PeftModel
)

# ============================================================
# Entraînement
# ============================================================

trainer.train()

# ============================================================
# Sauvegarder uniquement les poids LoRA
# ============================================================

trainer.model.save_pretrained(
    "TinyLlama-1.1B-qlora"
)

# Sauvegarder également le tokenizer
tokenizer.save_pretrained(
    "TinyLlama-1.1B-qlora"
)

print("Entraînement terminé.")
print("Modèle QLoRA sauvegardé dans : TinyLlama-1.1B-qlora")

Tokenizing train dataset:   0%|          | 0/3000 [00:00<?, ? examples/s]

KeyError: 'completion'

In [ ]:
from peft import AutoPeftModelForCausalLM
model = AutoPeftModelForCausalLM.from_pretrained(
 "TinyLlama-1.1B-qlora",
 low_cpu_mem_usage=True,
 device_map="auto",
)
# Merge LoRA and base model
merged_model = model.merge_and_unload()

In [ ]:
from transformers import pipeline
# Use our predefined prompt template
prompt = """<|user|>
Tell me something about Large Language Models.</s>
<|assistant|>
"""
# Run our instruction-tuned model
pipe = pipeline(task="text-generation", model=merged_model, tokenizer=tokenizer)
print(pipe(prompt)[0]["generated_text"])

In [ ]:
from datasets import load_dataset
def format_prompt(example):
 """Format the prompt to using the <|user|> template TinyLLama is using"""
 # Format answers
 system = "<|system|>\n" + example["system"] + "</s>\n"
 prompt = "<|user|>\n" + example["input"] + "</s>\n<|assistant|>\n"
 chosen = example["chosen"] + "</s>\n"
 rejected = example["rejected"] + "</s>\n"
 return {
 "prompt": system + prompt,
 "chosen": chosen,
 "rejected": rejected,
 }
# Apply formatting to the dataset and select relatively short answers
dpo_dataset = load_dataset(
 "argilla/distilabel-intel-orca-dpo-pairs", split="train"
)
dpo_dataset = dpo_dataset.filter(
 lambda r:
 r["status"] != "tie" and
 r["chosen_score"] >= 8 and
 not r["in_gsm8k_train"]
)
dpo_dataset = dpo_dataset.map(
 format_prompt, remove_columns=dpo_dataset.column_names
)
dpo_dataset

In [ ]:
from peft import AutoPeftModelForCausalLM
from transformers import BitsAndBytesConfig, AutoTokenizer
# 4-bit quantization configuration - Q in QLoRA
bnb_config = BitsAndBytesConfig(
 load_in_4bit=True, # Use 4-bit precision model loading
 bnb_4bit_quant_type="nf4", # Quantization type
 bnb_4bit_compute_dtype="float16", # Compute dtype
 bnb_4bit_use_double_quant=True, # Apply nested quantization
)
# Merge LoRA and base model
model = AutoPeftModelForCausalLM.from_pretrained(
 "TinyLlama-1.1B-qlora",
 low_cpu_mem_usage=True,
 device_map="auto",
 quantization_config=bnb_config,
)
merged_model = model.merge_and_unload()
# Load LLaMA tokenizer
model_name = "TinyLlama/TinyLlama-1.1B-intermediate-step-1431k-3T"
tokenizer = AutoTokenizer.from_pretrained(model_name, trust_remote_code=True)
tokenizer.pad_token = "<PAD>"
tokenizer.padding_side = "left"

In [ ]:
from peft import LoraConfig, prepare_model_for_kbit_training, get_peft_model
# Prepare LoRA configuration
peft_config = LoraConfig(
 lora_alpha=32, # LoRA Scaling
 lora_dropout=0.1, # Dropout for LoRA Layers
 r=64, # Rank
 bias="none",
 task_type="CAUSAL_LM",
 target_modules= # Layers to target
 ["k_proj", "gate_proj", "v_proj", "up_proj", "q_proj", "o_proj",
"down_proj"]
)
# prepare model for training
model = prepare_model_for_kbit_training(model)
model = get_peft_model(model, peft_config)

In [ ]:
from trl import DPOConfig
output_dir = "./results"
# Training arguments
training_arguments = DPOConfig(
 output_dir=output_dir,
 per_device_train_batch_size=2,
 gradient_accumulation_steps=4,
 optim="paged_adamw_32bit",
 learning_rate=1e-5,
 lr_scheduler_type="cosine",
 max_steps=200,
 logging_steps=10,
 fp16=True,
 gradient_checkpointing=True,
 warmup_ratio=0.1
)


In [ ]:
from trl import DPOTrainer
# Create DPO trainer
dpo_trainer = DPOTrainer(
 model,
 args=training_arguments,
 train_dataset=dpo_dataset,
 tokenizer=tokenizer,
 peft_config=peft_config,
 beta=0.1,
 max_prompt_length=512,
 max_length=512,
)
# Fine-tune model with DPO
dpo_trainer.train()
# Save adapter
dpo_trainer.model.save_pretrained("TinyLlama-1.1B-dpo-qlora")

In [ ]:
from peft import PeftModel
# Merge LoRA and base model
model = AutoPeftModelForCausalLM.from_pretrained(
 "TinyLla
 low_cpu_mem_usage=True,
 device_map="auto",
)
sft_model = model.merge_and_unload()
# Merge DPO LoRA and SFT model
dpo_model = PeftModel.from_pretrained(
 sft_model,
 "TinyLlama-1.1B-dpo-qlora",
 device_map="auto",
)
dpo_model = dpo_model.merge_and_unload()